In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install timm

In [ ]:
# ============================================================
# KNOWLEDGE DISTILLATION: ResNet18 (Teacher) → ShuffleNet (Student)
# Coin Classification Task
# ============================================================
print('start')
import os
import copy
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, classification_report
)

# ============================================================
# 1. DATASET SETUP (your existing code)
# ============================================================

IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

input_path = "/kaggle/input/datasets/zarinsaimaroza/coin-dataset/Final_Balanced_Coin_Dataset/"
working_path = "/kaggle/working/coin_dataset"

if not os.path.exists(working_path):
    shutil.copytree(input_path, working_path)

train_dir = os.path.join(working_path, "train")
val_dir   = os.path.join(working_path, "valid")
test_dir  = os.path.join(working_path, "test")

label_map = {"1f": "0", "2f": "1", "5f": "2", "1b": "3", "2b": "4", "5b": "5"}

def move_unlabeled_images(base_dir):
    unlabeled_path = os.path.join(base_dir, "unlabeled")
    if not os.path.exists(unlabeled_path):
        return
    moved_count = 0
    for img_name in os.listdir(unlabeled_path):
        prefix = img_name.split("_")[0]
        if prefix in label_map:
            target_class = label_map[prefix]
            target_dir = os.path.join(base_dir, target_class)
            os.makedirs(target_dir, exist_ok=True)
            shutil.move(os.path.join(unlabeled_path, img_name),
                        os.path.join(target_dir, img_name))
            moved_count += 1
    print(f"{base_dir} → moved {moved_count} images")
    if len(os.listdir(unlabeled_path)) == 0:
        os.rmdir(unlabeled_path)

for split_dir in [train_dir, val_dir, test_dir]:
    move_unlabeled_images(split_dir)
print("✅ Unlabeled images moved successfully!")

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(val_dir,   transform=val_transform)
test_dataset  = datasets.ImageFolder(test_dir,  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

NUM_CLASSES   = len(train_dataset.classes)
CLASS_NAMES   = train_dataset.classes

print(f"Train images      : {len(train_dataset)}")
print(f"Validation images : {len(val_dataset)}")
print(f"Test images       : {len(test_dataset)}")
print(f"Classes ({NUM_CLASSES})    : {CLASS_NAMES}")

# ============================================================
# 2. DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🖥️  Using device: {device}")

# ============================================================
# 3. MODEL DEFINITIONS
# ============================================================

def build_teacher(num_classes):
    """ResNet-18 as Teacher"""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

def build_student(num_classes):
    """EfficientNetV2-B3 (TF variant) as Student"""
    model = timm.create_model(
        'tf_efficientnetv2_b3',
        pretrained=True,
        num_classes=num_classes
    )
    return model.to(device)

teacher = build_teacher(NUM_CLASSES)
student = build_student(NUM_CLASSES)

print(f"\n📐 Teacher params : {sum(p.numel() for p in teacher.parameters()):,}")
print(f"📐 Student params : {sum(p.numel() for p in student.parameters()):,}")

# ============================================================
# 4. KNOWLEDGE DISTILLATION LOSS
# ============================================================

class DistillationLoss(nn.Module):
    """
    Combined loss = alpha * CrossEntropy(student, labels)
                  + (1 - alpha) * KLDiv(student_soft, teacher_soft) * T^2
    """
    def __init__(self, temperature=4.0, alpha=0.4):
        super().__init__()
        self.T     = temperature
        self.alpha = alpha
        self.ce    = nn.CrossEntropyLoss()

    def forward(self, student_logits, teacher_logits, labels):
        # Hard label loss
        loss_ce = self.ce(student_logits, labels)

        # Soft label loss (KL divergence)
        student_soft = F.log_softmax(student_logits / self.T, dim=1)
        teacher_soft = F.softmax(teacher_logits   / self.T, dim=1)
        loss_kd = F.kl_div(student_soft, teacher_soft, reduction="batchmean") * (self.T ** 2)

        return self.alpha * loss_ce + (1 - self.alpha) * loss_kd

# ============================================================
# 5. TRAINING UTILITIES
# ============================================================

class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_loss  = None
        self.counter    = 0
        self.best_state = None

    def step(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
        return self.counter >= self.patience   # True → stop

    def restore_best(self, model):
        model.load_state_dict(self.best_state)


def evaluate(model, loader):
    """Returns (avg_loss, accuracy) on a dataloader."""
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out  = model(imgs)
            loss = criterion(out, labels)
            total_loss += loss.item() * imgs.size(0)
            preds       = out.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
    return total_loss / total, correct / total

# ============================================================
# 6. TEACHER TRAINING
# ============================================================

EPOCHS        = 100
PATIENCE      = 12
LR_TEACHER    = 1e-3
LR_STUDENT    = 1e-3
TEMPERATURE   = 4.0
ALPHA         = 0.4           # weight for hard-label loss

print("\n" + "="*55)
print("  PHASE 1 — Training Teacher (ResNet-18)")
print("="*55)

teacher_optimizer = AdamW(teacher.parameters(), lr=LR_TEACHER, weight_decay=1e-4)
teacher_scheduler = CosineAnnealingLR(teacher_optimizer, T_max=EPOCHS)
teacher_criterion = nn.CrossEntropyLoss()
teacher_es        = EarlyStopping(patience=PATIENCE)

teacher_train_losses, teacher_val_losses   = [], []
teacher_train_accs,   teacher_val_accs     = [], []

for epoch in range(1, EPOCHS + 1):
    # --- train ---
    teacher.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        teacher_optimizer.zero_grad()
        out  = teacher(imgs)
        loss = teacher_criterion(out, labels)
        loss.backward()
        teacher_optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct      += (out.argmax(1) == labels).sum().item()
        total        += imgs.size(0)
    teacher_scheduler.step()

    t_loss = running_loss / total
    t_acc  = correct / total

    # --- validate ---
    v_loss, v_acc = evaluate(teacher, val_loader)

    teacher_train_losses.append(t_loss);  teacher_val_losses.append(v_loss)
    teacher_train_accs.append(t_acc);     teacher_val_accs.append(v_acc)

    print(f"[Teacher] Epoch {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {t_loss:.4f}  Acc: {t_acc:.4f} | "
          f"Val Loss: {v_loss:.4f}  Acc: {v_acc:.4f}")

    if teacher_es.step(v_loss, teacher):
        print(f"  ⏹  Early stopping at epoch {epoch}")
        break

teacher_es.restore_best(teacher)
print("✅ Teacher training complete. Best val loss restored.")

# ============================================================
# 7. STUDENT TRAINING (KNOWLEDGE DISTILLATION)
# ============================================================

print("\n" + "="*55)
print("  PHASE 2 — Training Student (ShuffleNet) via KD")
print("="*55)

student_optimizer = AdamW(student.parameters(), lr=LR_STUDENT, weight_decay=1e-4)
student_scheduler = CosineAnnealingLR(student_optimizer, T_max=EPOCHS)
kd_criterion      = DistillationLoss(temperature=TEMPERATURE, alpha=ALPHA)
student_es        = EarlyStopping(patience=PATIENCE)

student_train_losses, student_val_losses = [], []
student_train_accs,   student_val_accs   = [], []

teacher.eval()   # freeze teacher in eval mode

for epoch in range(1, EPOCHS + 1):
    # --- train ---
    student.train()
    running_loss, correct, total = 0.0, 0, 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        with torch.no_grad():
            teacher_logits = teacher(imgs)

        student_optimizer.zero_grad()
        student_logits = student(imgs)
        loss = kd_criterion(student_logits, teacher_logits, labels)
        loss.backward()
        student_optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        correct      += (student_logits.argmax(1) == labels).sum().item()
        total        += imgs.size(0)

    student_scheduler.step()

    s_loss = running_loss / total
    s_acc  = correct / total

    # --- validate ---
    v_loss, v_acc = evaluate(student, val_loader)

    student_train_losses.append(s_loss);  student_val_losses.append(v_loss)
    student_train_accs.append(s_acc);     student_val_accs.append(v_acc)

    print(f"[Student] Epoch {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {s_loss:.4f}  Acc: {s_acc:.4f} | "
          f"Val Loss: {v_loss:.4f}  Acc: {v_acc:.4f}")

    if student_es.step(v_loss, student):
        print(f"  ⏹  Early stopping at epoch {epoch}")
        break

student_es.restore_best(student)
print("✅ Student training complete. Best val loss restored.")

# ============================================================
# 8. SAVE MODELS
# ============================================================

torch.save(teacher.state_dict(), "/kaggle/working/teacher_resnet18.pth")
torch.save(student.state_dict(), "/kaggle/working/student_shufflenet.pth")
print("💾 Models saved.")

# ============================================================
# 9. LEARNING CURVES
# ============================================================

def plot_curves(train_losses, val_losses, train_accs, val_accs, title_prefix, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs_ran = range(1, len(train_losses) + 1)

    axes[0].plot(epochs_ran, train_losses, label="Train Loss", linewidth=2)
    axes[0].plot(epochs_ran, val_losses,   label="Val Loss",   linewidth=2)
    axes[0].set_title(f"{title_prefix} — Loss Curve")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs_ran, train_accs, label="Train Acc", linewidth=2)
    axes[1].plot(epochs_ran, val_accs,   label="Val Acc",   linewidth=2)
    axes[1].set_title(f"{title_prefix} — Accuracy Curve")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"📊 Saved: {save_path}")

plot_curves(teacher_train_losses, teacher_val_losses,
            teacher_train_accs,   teacher_val_accs,
            "Teacher (ResNet-18)",
            "/kaggle/working/teacher_curves.png")

plot_curves(student_train_losses, student_val_losses,
            student_train_accs,   student_val_accs,
            "Student (ShuffleNet) — KD",
            "/kaggle/working/student_curves.png")

# ============================================================
# 10. EVALUATION FUNCTION (full metrics)
# ============================================================

def full_evaluation(model, loader, model_name="Model"):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(device)
            preds  = model(imgs).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    acc       = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
    recall    = recall_score(all_labels, all_preds,    average="weighted", zero_division=0)
    f1        = f1_score(all_labels, all_preds,        average="weighted", zero_division=0)

    print(f"\n{'='*55}")
    print(f"  {model_name} — Evaluation Results")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"\nClassification Report:\n")
    print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, zero_division=0))

    return all_preds, all_labels, acc, precision, recall, f1

# ============================================================
# 11. CONFUSION MATRIX PLOT
# ============================================================

def plot_confusion_matrix(labels, preds, class_names, title, save_path):
    cm = confusion_matrix(labels, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Raw counts
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title(f"{title} — Counts")
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

    # Normalised
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=axes[1])
    axes[1].set_title(f"{title} — Normalised")
    axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"📊 Saved: {save_path}")

# ============================================================
# 12. RUN EVALUATIONS
# ============================================================

print("\n🔍 Evaluating TEACHER on validation set ...")
t_val_preds, t_val_labels, *_ = full_evaluation(teacher, val_loader, "Teacher (ResNet-18) — Val")
plot_confusion_matrix(t_val_labels, t_val_preds, CLASS_NAMES,
                      "Teacher Val", "/kaggle/working/cm_teacher_val.png")

print("\n🔍 Evaluating TEACHER on test set ...")
t_test_preds, t_test_labels, *_ = full_evaluation(teacher, test_loader, "Teacher (ResNet-18) — Test")
plot_confusion_matrix(t_test_labels, t_test_preds, CLASS_NAMES,
                      "Teacher Test", "/kaggle/working/cm_teacher_test.png")

print("\n🔍 Evaluating STUDENT on validation set ...")
s_val_preds, s_val_labels, *_ = full_evaluation(student, val_loader, "Student (ShuffleNet) — Val")
plot_confusion_matrix(s_val_labels, s_val_preds, CLASS_NAMES,
                      "Student Val", "/kaggle/working/cm_student_val.png")

print("\n🔍 Evaluating STUDENT on test set ...")
s_test_preds, s_test_labels, s_acc, s_prec, s_rec, s_f1 = full_evaluation(student, test_loader, "Student (ShuffleNet) — Test")
plot_confusion_matrix(s_test_labels, s_test_preds, CLASS_NAMES,
                      "Student Test", "/kaggle/working/cm_student_test.png")

# ============================================================
# 13. TEACHER vs STUDENT COMPARISON TABLE
# ============================================================

def get_metrics(model, loader):
    model.eval()
    preds_list, labels_list = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            preds_list.extend(model(imgs.to(device)).argmax(1).cpu().numpy())
            labels_list.extend(labels.numpy())
    p, l = np.array(preds_list), np.array(labels_list)
    return {
        "Accuracy" : accuracy_score(l, p),
        "Precision": precision_score(l, p, average="weighted", zero_division=0),
        "Recall"   : recall_score(l, p, average="weighted", zero_division=0),
        "F1"       : f1_score(l, p, average="weighted", zero_division=0),
    }

print("\n" + "="*55)
print("  📋  TEACHER vs STUDENT — Final Comparison")
print("="*55)

for split_name, loader in [("Validation", val_loader), ("Test", test_loader)]:
    tm = get_metrics(teacher, loader)
    sm = get_metrics(student, loader)
    print(f"\n  [{split_name} Set]")
    print(f"  {'Metric':<12} {'Teacher':>10} {'Student':>10}")
    print(f"  {'-'*34}")
    for k in tm:
        print(f"  {k:<12} {tm[k]:>10.4f} {sm[k]:>10.4f}")

# ============================================================
# 14. TRAIN SET ACCURACY (FINAL EPOCH SNAPSHOT)
# ============================================================

print("\n📌 Final Train Accuracy:")
print(f"   Teacher : {teacher_train_accs[-1]:.4f}")
print(f"   Student : {student_train_accs[-1]:.4f}")
print("\n✅ All done! Check /kaggle/working/ for saved models and plots.")

start
/kaggle/working/coin_dataset/train → moved 1096 images
/kaggle/working/coin_dataset/valid → moved 161 images
/kaggle/working/coin_dataset/test → moved 305 images
✅ Unlabeled images moved successfully!
Train images      : 6221
Validation images : 897
Test images       : 1788
Classes (6)    : ['0', '1', '2', '3', '4', '5']

🖥️  Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 208MB/s]


model.safetensors:   0%|          | 0.00/57.9M [00:00<?, ?B/s]


📐 Teacher params : 11,179,590
📐 Student params : 12,830,628

  PHASE 1 — Training Teacher (ResNet-18)
[Teacher] Epoch 001/100 | Train Loss: 0.5134  Acc: 0.8203 | Val Loss: 0.7337  Acc: 0.7793
[Teacher] Epoch 002/100 | Train Loss: 0.2057  Acc: 0.9322 | Val Loss: 0.0963  Acc: 0.9654
[Teacher] Epoch 003/100 | Train Loss: 0.1635  Acc: 0.9453 | Val Loss: 0.1992  Acc: 0.9309
[Teacher] Epoch 004/100 | Train Loss: 0.0946  Acc: 0.9674 | Val Loss: 0.1026  Acc: 0.9610
[Teacher] Epoch 005/100 | Train Loss: 0.0945  Acc: 0.9693 | Val Loss: 0.2664  Acc: 0.9621
[Teacher] Epoch 006/100 | Train Loss: 0.0790  Acc: 0.9757 | Val Loss: 0.0322  Acc: 0.9855
[Teacher] Epoch 007/100 | Train Loss: 0.0699  Acc: 0.9757 | Val Loss: 0.2376  Acc: 0.9253
[Teacher] Epoch 008/100 | Train Loss: 0.1237  Acc: 0.9624 | Val Loss: 0.0498  Acc: 0.9833
[Teacher] Epoch 009/100 | Train Loss: 0.0506  Acc: 0.9830 | Val Loss: 0.0650  Acc: 0.9788
[Teacher] Epoch 010/100 | Train Loss: 0.0267  Acc: 0.9910 | Val Loss: 0.0780  Acc: 0.97